# Cascad — Hugging Face attribution baseline

This notebook runs one frozen local model at a time on Kaggle or Google Colab. Select a GPU runtime before starting. Run `qwen3-4b` first; use a fresh session for `mistral-7b` if disk space is limited.

In [2]:
import importlib
import os
import pathlib
import subprocess
import sys

REPO_URL = os.environ.get("CASCAD_REPO_URL", "https://github.com/elom354/cascad.git")
MODEL_ALIAS = os.environ.get("CASCAD_HF_MODEL", "qwen3-4b")
assert MODEL_ALIAS in {"qwen3-4b", "mistral-7b"}

base = pathlib.Path("/kaggle/working" if pathlib.Path("/kaggle/working").exists() else "/content")
repo = base / "Cascad"
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
os.chdir(repo)
print({"repository": str(repo), "model": MODEL_ALIAS})

{'repository': '/content/Cascad', 'model': 'qwen3-4b'}


In [3]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[huggingface]"], check=True)
src = str(repo / "src")
os.environ["PYTHONPATH"] = src + os.pathsep + os.environ.get("PYTHONPATH", "")
if src not in sys.path:
    sys.path.insert(0, src)
importlib.import_module("cascad")
torch = importlib.import_module("torch")
assert torch.cuda.is_available(), "Enable a GPU accelerator in the notebook settings"
print({"torch": torch.__version__, "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0)})

{'torch': '2.11.0+cu128', 'cuda': '12.8', 'gpu': 'Tesla T4'}


In [7]:
try:
    from google.colab import userdata

    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
except Exception as exc:
    print("Impossible de charger HF_TOKEN :", exc)

print("HF token configured:", bool(os.environ.get("HF_TOKEN")))

HF token configured: True


In [8]:
output = base / f"cascad-huggingface-{MODEL_ALIAS}"
command = [
    sys.executable,
    "scripts/run_huggingface_attribution.py",
    "--models", MODEL_ALIAS,
    "--quantization", "4bit",
    "--out", str(output),
]
print(" ".join(command))
process = subprocess.Popen(
    command,
    cwd=repo,
    env=os.environ.copy(),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
tail = []
assert process.stdout is not None
for line in process.stdout:
    print(line, end="")
    tail.append(line)
    tail = tail[-80:]
return_code = process.wait()
if return_code:
    raise RuntimeError(
        f"Hugging Face runner failed with exit code {return_code}.\n"
        + "".join(tail)
    )

/usr/bin/python3 scripts/run_huggingface_attribution.py --models qwen3-4b --quantization 4bit --out /content/cascad-huggingface-qwen3-4b
2026-07-28 06:44:50.119408: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.

Fetching 3 files: 100%|██████████| 3/3 [01:11<00:00, 23.81s/it]

Loading checkpoint shards: 100%|██████████| 3/3 [00:33<00:00, 11.29s/it]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[qwen3-4b] 1/200 controlled--full-tools-persistent-memory--mono_step--explicit_tool_error--000 completed
[qwen3-4b] 2/200 controlled--full-tools-persistent-memory--mono_step--explicit_tool_error--001 completed
[qwen3-4b] 3/200 controlled--full-tools-pe

RuntimeError: Hugging Face runner failed with exit code 1.
[qwen3-4b] 134/200 controlled--local-core-no-memory--mono_step--incorrect_numeric_value--008 completed
[qwen3-4b] 135/200 controlled--local-core-no-memory--mono_step--incorrect_numeric_value--009 completed
[qwen3-4b] 136/200 controlled--local-core-no-memory--mono_step--incorrect_numeric_value--010 completed
[qwen3-4b] 137/200 controlled--local-core-no-memory--mono_step--incorrect_numeric_value--011 completed
[qwen3-4b] 138/200 controlled--local-core-no-memory--mono_step--incorrect_numeric_value--012 completed
[qwen3-4b] 139/200 controlled--local-core-no-memory--mono_step--incorrect_numeric_value--013 completed
[qwen3-4b] 140/200 controlled--local-core-no-memory--mono_step--incorrect_numeric_value--014 completed
[qwen3-4b] 141/200 controlled--local-core-no-memory--mono_step--incorrect_numeric_value--015 completed
[qwen3-4b] 142/200 controlled--local-core-no-memory--mono_step--incorrect_numeric_value--016 completed
[qwen3-4b] 143

In [9]:
import json
import shutil

summary_path = output / "summary.json"
assert summary_path.is_file(), "The runner did not produce summary.json"
summary = json.loads(summary_path.read_text())
print(json.dumps(summary, indent=2))
archive = shutil.make_archive(str(output), "zip", output)
print("Download this archive before closing the session:", archive)

AssertionError: The runner did not produce summary.json